In [1]:
"""
Kaggle 제출용 매핑 검증 스크립트
"""
import pandas as pd
import yaml

# 1. K-code → dl_idx 패턴 확인
df = pd.read_csv('D:/pill_detection/eda/train.csv')
sample = df['img_dl_mapping_code'].drop_duplicates().head(10).tolist()

print('=== K-code 샘플 10개 (dl_idx 추정값) ===')
for k in sample:
    n = int(k.replace('K-', ''))
    print(f'  {k} -> dl_idx 후보: {n}')
print(f'\n총 unique K-code: {df["img_dl_mapping_code"].nunique()}')

# 2. pill.yaml class_idx → K-code 매핑
print('\n' + '=' * 60)
y = yaml.safe_load(open('D:/pill_detection/data/dataset_yolo/pill.yaml', 'r', encoding='utf-8'))
name_to_kcode = df.drop_duplicates('img_dl_name').set_index('img_dl_name')['img_dl_mapping_code'].to_dict()

print('\n=== pill.yaml class_idx -> K-code 매핑 샘플 5개 ===')
for i in range(5):
    name = y['names'][i]
    kcode = name_to_kcode.get(name, '???')
    print(f'  idx={i} | {name} -> {kcode}')

match = sum(1 for i in range(118) if y['names'][i] in name_to_kcode)
print(f'\nyaml 118개 중 train.csv 매칭: {match}/118')

# 3. 최종 매핑 테이블 생성 (class_idx -> dl_idx)
print('\n' + '=' * 60)
print('\n=== 최종 매핑 테이블 (class_idx -> dl_idx) ===')
mapping = {}
for i in range(118):
    name = y['names'][i]
    kcode = name_to_kcode.get(name)
    if kcode:
        dl_idx = int(kcode.replace('K-', ''))
        mapping[i] = dl_idx
    else:
        print(f'  ❌ idx={i} | {name} -> 매칭 실패')

print(f'\n매핑 성공: {len(mapping)}/118')
print('첫 5개:', dict(list(mapping.items())[:5]))

=== K-code 샘플 10개 (dl_idx 추정값) ===
  K-001900 -> dl_idx 후보: 1900
  K-016548 -> dl_idx 후보: 16548
  K-019607 -> dl_idx 후보: 19607
  K-033009 -> dl_idx 후보: 33009
  K-021771 -> dl_idx 후보: 21771
  K-027926 -> dl_idx 후보: 27926
  K-024850 -> dl_idx 후보: 24850
  K-029345 -> dl_idx 후보: 29345
  K-002483 -> dl_idx 후보: 2483
  K-019552 -> dl_idx 후보: 19552

총 unique K-code: 118


=== pill.yaml class_idx -> K-code 매핑 샘플 5개 ===
  idx=0 | 마그밀정(수산화마그네슘) -> K-000250
  idx=1 | 게보린정 300mg/PTP -> K-000573
  idx=2 | 알마겔정(알마게이트)(수출명:유한가스트라겔정) -> K-001866
  idx=3 | 보령부스파정 5mg -> K-001900
  idx=4 | 뮤테란캡슐 100mg -> K-002483

yaml 118개 중 train.csv 매칭: 118/118


=== 최종 매핑 테이블 (class_idx -> dl_idx) ===

매핑 성공: 118/118
첫 5개: {0: 250, 1: 573, 2: 1866, 3: 1900, 4: 2483}


In [2]:
"""
class_idx -> dl_idx 매핑 JSON 저장
"""
import pandas as pd
import yaml
import json

df = pd.read_csv('D:/pill_detection/eda/train.csv')
y = yaml.safe_load(open('D:/pill_detection/data/dataset_yolo/pill.yaml', 'r', encoding='utf-8'))
name_to_kcode = df.drop_duplicates('img_dl_name').set_index('img_dl_name')['img_dl_mapping_code'].to_dict()

mapping = {}
for i in range(118):
    name = y['names'][i]
    kcode = name_to_kcode[name]
    dl_idx = int(kcode.replace('K-', ''))
    mapping[i] = {'name': name, 'kcode': kcode, 'dl_idx': dl_idx}

# JSON 저장
out = 'D:/pill_detection/eda/class_idx_to_dl_idx.json'
with open(out, 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)

print(f'저장: {out}')
print(f'118개 매핑 완료')
print(f'\n샘플 3개:')
for k in list(mapping.keys())[:3]:
    print(f'  {k}: {mapping[k]}')

저장: D:/pill_detection/eda/class_idx_to_dl_idx.json
118개 매핑 완료

샘플 3개:
  0: {'name': '마그밀정(수산화마그네슘)', 'kcode': 'K-000250', 'dl_idx': 250}
  1: {'name': '게보린정 300mg/PTP', 'kcode': 'K-000573', 'dl_idx': 573}
  2: {'name': '알마겔정(알마게이트)(수출명:유한가스트라겔정)', 'kcode': 'K-001866', 'dl_idx': 1866}
